In [2]:
import numpy as np
from collections import Counter
from ase.build import surface, make_supercell
from ase.io import write, read
from ase.visualize import view
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.io.lammps.data import LammpsData
from pymatgen.core.surface import Slab
import os

os.makedirs("slabs", exist_ok=True)

ld = LammpsData.from_file('bulk_structure/mg_unitcell_relaxed.data', atom_style='atomic')
alloy = AseAtomsAdaptor().get_atoms(ld.structure)
print(f"Bulk: {len(alloy)} atoms, {alloy.get_chemical_formula()}")
print(f"Cell: {np.round(alloy.cell.lengths(), 4)} A")

Bulk: 2 atoms, Mg2
Cell: [3.2091 3.2091 5.1969] A


In [5]:
# (10-11) = (1,0,1) in 3-index
slab = surface(alloy, (1, 0, 1), 15, vacuum=8)

print(f"Atoms: {len(slab)}")
print(f"Cell: {np.round(slab.cell.lengths(), 3)} A")

area = np.linalg.norm(np.cross(slab.cell[0], slab.cell[1]))
print(f"Surface area: {area:.2f} A^2")

a_len = np.linalg.norm(slab.cell[0])
b_len = np.linalg.norm(slab.cell[1])
print(f"In-plane: a={a_len:.2f}, b={b_len:.2f} A")

na = max(1, int(np.ceil(9.0 / a_len)))
nb = max(1, int(np.ceil(9.0 / b_len)))
if na > 1 or nb > 1:
    print(f"Small area — making {na}x{nb} supercell")
    slab = make_supercell(slab, [[na,0,0],[0,nb,0],[0,0,1]])
    area = np.linalg.norm(np.cross(slab.cell[0], slab.cell[1]))
    print(f"New: {len(slab)} atoms, area={area:.2f} A^2")

z = slab.get_positions()[:, 2]
thick = z.max() - z.min()
print(f"Thickness: {thick:.2f} A")

ps = AseAtomsAdaptor().get_structure(slab)
slab_obj = Slab(ps.lattice, ps.species, ps.frac_coords,
    miller_index=(1,0,1), oriented_unit_cell=ps, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])
print(f"Symmetric: {slab_obj.is_symmetric()}")

Atoms: 30
Cell: [ 6.108  3.209 52.352] A
Surface area: 18.91 A^2
In-plane: a=6.11, b=3.21 A
Small area — making 2x3 supercell
New: 180 atoms, area=113.47 A^2
Thickness: 36.35 A
Symmetric: True


In [6]:
view(slab)

<Popen: returncode: None args: ['C:\\Users\\Kai Savage\\anaconda3\\python.ex...>

In [7]:
ps = AseAtomsAdaptor().get_structure(slab)
ld_out = LammpsData.from_structure(ps, atom_style='atomic')
ld_out.write_file("slabs/Mg_1011_8L.data")

z = slab.get_positions()[:, 2]
thick = z.max() - z.min()
area = np.linalg.norm(np.cross(slab.cell[0], slab.cell[1]))

print(f"Saved: slabs/Mg_1011_8L.data")
print(f"  Atoms: {len(slab)}")
print(f"  Thickness: {thick:.2f} A")
print(f"  Area: {area:.2f} A^2")

Saved: slabs/Mg_1011_8L.data
  Atoms: 180
  Thickness: 36.35 A
  Area: 113.47 A^2
